<a href="https://colab.research.google.com/github/ylcoder/random-walk/blob/main/fed_betting_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fetch Event ID

### Subtask:
Make a GET request to `https://gamma-api.polymarket.com/events?slug=<event-slug>` to retrieve the `id` for the specified event. This step will also verify if this specific endpoint requires authentication, and if so, how to address it.


**Reasoning**:
To investigate the Polymarket API for event details and authentication requirements, I will construct a GET request to the specified endpoint with an event slug and handle both successful JSON responses and error text responses.



In [ ]:
import requests
import json

# Reusing base_url from previous steps
# base_url = "https://gamma-api.polymarket.com/"

event_slug = 'fed-decision-in-march-885'

# Construct the API endpoint URL
endpoint_url = f'{base_url}events?slug={event_slug}'

try:
    response = requests.get(endpoint_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    api_response_json = response.json()
    print(f"API Response for event '{event_slug}':")
    print(json.dumps(api_response_json, indent=2))
except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
    print("Raw error response:")
    print(response.text)
except Exception as err:
    print(f"Other error occurred: {err}")

## Fetch Trade Data

### Subtask:
Using the obtained event ID, make a GET request to `https://data-api.polymarket.com/trades?eventId=<id>&limit=10000&offset=0` to extract all trade data. We will also investigate how to filter by 'last N days' if the API supports it, or devise an alternative approach to filter locally after fetching.


**Reasoning**:
I will extract the event ID from the `api_response_json` and then construct a new URL to fetch trade data using a GET request to the specified Polymarket API endpoint.



In [36]:
import requests
import json

# 1. Extract the 'id' for the event from the api_response_json variable
# Assuming api_response_json is a list containing dictionaries, and the first dictionary has the 'id'
if api_response_json and isinstance(api_response_json, list) and len(api_response_json) > 0:
    event_id = api_response_json[0].get('id')
    print(f"Extracted Event ID: {event_id}")
else:
    event_id = None
    print("Could not extract event ID from api_response_json.")

trade_api_response_jsons = []

if event_id:
    # 2. Construct the API endpoint URL for fetching trade data
    trade_data_base_url = "https://data-api.polymarket.com/trades"
    limit = 3000
    offset = 0

    # 3. Make a GET request to the constructed trade data API endpoint.
    try:
        while limit > 0:
            trade_data_endpoint_url = f'{trade_data_base_url}?eventId={event_id}&limit=1000&offset={offset}'
            print(f"Fetching trade data from: {trade_data_endpoint_url}")

            response = requests.get(trade_data_endpoint_url)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            trade_api_response_jsons.append(response.json())
            offset += 1000
            limit = limit - 1000
            # print("Polymarket Trade Data API Response:")
            # print(json.dumps(trade_api_response_json, indent=2))
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print("Raw error response:")
        print(response.text)
    except Exception as err:
        print(f"Other error occurred: {err}")
else:
    print("Cannot fetch trade data without a valid event ID.")

Extracted Event ID: 67284
Fetching trade data from: https://data-api.polymarket.com/trades?eventId=67284&limit=1000&offset=0
Fetching trade data from: https://data-api.polymarket.com/trades?eventId=67284&limit=1000&offset=1000
Fetching trade data from: https://data-api.polymarket.com/trades?eventId=67284&limit=1000&offset=2000


# Task
Implement a function to parse the raw API response from `trade_api_response_json`, extracting `name` (as `user_id`), `side`, `size`, `outcome`, `title`, and `price`, and calculating the total trade amount for each trade.

## Implement Data Parsing and Extraction

### Subtask:
Create a function to parse the raw API response data, extract specified fields, and calculate the total trade amount for each trade.


**Reasoning**:
I will define a Python function `parse_trade_data` to process the raw trade data. This function will iterate through each trade, extract the required fields, calculate the total trade amount, and return a list of cleaned dictionaries.



In [37]:
import datetime
import pytz

def seconds_to_est(seconds):
    """
    Converts seconds since the epoch to a date and time string in EST.

    Args:
        seconds (int/float): The number of seconds since the epoch (UTC).

    Returns:
        str: The formatted date and time string in EST.
    """
    # 1. Convert seconds to a UTC datetime object
    # The fromtimestamp() function without an explicit timezone assumes the system's local time,
    # so we explicitly set it to UTC first.
    utc_datetime = datetime.datetime.fromtimestamp(seconds, tz=pytz.utc)

    # 2. Define the target timezone (EST)
    # Use 'America/New_York' for the US Eastern time zone, which handles both EST and EDT.
    est_timezone = pytz.timezone('America/New_York')

    # 3. Convert the UTC datetime object to the EST timezone
    est_datetime = utc_datetime.astimezone(est_timezone)

    # 4. Format the datetime object as a string (e.g., "Sunday, January 29, 2017 08:30:00 EST")
    # A full list of format codes is available in the [Python documentation](https://docs.python.org).
    formatted_time = est_datetime.strftime("%A, %B %d, %Y %I:%M:%S %Z")

    return formatted_time

# Example usage:
# Example timestamp for January 29, 2017 13:30:00 UTC (08:30:00 EST)
timestamp_seconds = 1772140056

est_time_str = seconds_to_est(timestamp_seconds)

print(f"Seconds input: {timestamp_seconds}")
print(f"Date and Time in EST: {est_time_str}")


Seconds input: 1772140056
Date and Time in EST: Thursday, February 26, 2026 04:07:36 EST


In [39]:
def parse_trade_data(raw_trades_data):
    """
    Parses raw Polymarket trade data, extracts specified fields, and calculates total trade amount.

    Args:
        raw_trades_data (list): A list of dictionaries, where each dictionary represents a raw trade.

    Returns:
        list: A list of dictionaries, each containing the parsed and calculated trade information.
    """
    processed_data = []
    for trade in raw_trades_data:
        try:
            user_id = trade.get('name')
            side = trade.get('side')
            size = trade.get('size')
            outcome = trade.get('outcome')
            title = trade.get('title')
            price = trade.get('price')
            timestamp = seconds_to_est(int(trade.get('timestamp')))

            # Calculate total trade amount
            total_trade_amount = float(size) * float(price) if size is not None and price is not None else 0.0

            processed_trade = {
                'user_id': user_id,
                'trade_direction': side,
                'shares': size,
                'option_yes_no': outcome,
                'option_type': title,
                'price': price,
                'total_trade_amount': total_trade_amount,
                'timestamp': timestamp
            }
            processed_data.append(processed_trade)
        except Exception as e:
            print(f"Error processing trade: {trade}. Error: {e}")
            # Optionally, append a placeholder or skip this trade
            continue
    return processed_data

print("The `parse_trade_data` function has been defined.")

The `parse_trade_data` function has been defined.


**Reasoning**:
The `parse_trade_data` function has been defined. Now, I will call this function with the `trade_api_response_json` to process the raw trade data and store the results in a new variable.



In [40]:
processed_trades_data = []
for trade_api_response_json in trade_api_response_jsons:
  processed_trades_data += parse_trade_data(trade_api_response_json)

print(f"Processed {len(processed_trades_data)} trades.")
# Display the first 5 processed trades to verify the output structure
print("First 2 Processed Trades:")
print(json.dumps(processed_trades_data[-1:], indent=2))

Processed 3000 trades.
First 2 Processed Trades:
[
  {
    "user_id": "Trng16743393",
    "trade_direction": "BUY",
    "shares": 8.6432,
    "option_yes_no": "No",
    "option_type": "Will the Fed decrease interest rates by 50+ bps after the March 2026 meeting?",
    "price": 0.995,
    "total_trade_amount": 8.599984000000001,
    "timestamp": "Thursday, February 26, 2026 02:35:28 EST"
  }
]


In [41]:
len(processed_trades_data)

3000

## Write to CSV

### Subtask:
Develop a routine to write the extracted and parsed data into a CSV file. This will involve creating a Pandas DataFrame from the extracted data and then saving it to a CSV file with appropriate headers.


**Reasoning**:
To write the processed data to a CSV file, I will first import the pandas library, then convert the `processed_trades_data` list into a pandas DataFrame, and finally save this DataFrame to 'polymarket_trades.csv' without including the DataFrame index.



In [42]:
import pandas as pd

# Create a Pandas DataFrame from the processed_trades_data
df_trades = pd.DataFrame(processed_trades_data)

# Save the DataFrame to a CSV file named 'polymarket_trades.csv'
df_trades.to_csv('polymarket_trades.csv', index=False)

print("Trade data successfully saved to 'polymarket_trades.csv'")
print("First 5 rows of the DataFrame:")
print(df_trades.head())

Trade data successfully saved to 'polymarket_trades.csv'
First 5 rows of the DataFrame:
        user_id trade_direction  shares option_yes_no  \
0  minhdang1991             BUY  7.3164            No   
1     chilenheo             BUY  7.8390            No   
2      QuynBch1             BUY  5.2864            No   
3      thief148             BUY  6.7870            No   
4      63dtoxm0            SELL  4.2000           Yes   

                                         option_type  price  \
0  Will the Fed decrease interest rates by 50+ bp...  0.995   
1  Will the Fed decrease interest rates by 50+ bp...  0.995   
2  Will the Fed decrease interest rates by 50+ bp...  0.995   
3  Will the Fed increase interest rates by 25+ bp...  0.996   
4  Will there be no change in Fed interest rates ...  0.973   

   total_trade_amount                                 timestamp  
0            7.279818  Thursday, February 26, 2026 05:01:00 EST  
1            7.799805  Thursday, February 26, 2026 05:00:2

## Integrate and Test

### Subtask:
Combine all the developed routines into a single script. Test the complete pipeline with a sample event ID and number of days to ensure data is fetched, parsed, and written to the CSV correctly. Verify the CSV content against expected output.


## Summary:

### Data Analysis Key Findings

*   A Python function, `parse_trade_data`, was successfully implemented to parse raw API response data, extracting and renaming fields such as `name` to `user_id`, `side` to `trade_direction`, `size` to `shares`, `outcome` to `option_yes_no`, `title` to `option_type`, and `price`.
*   The `parse_trade_data` function accurately calculated the `total_trade_amount` for each trade by multiplying `size` and `price`, including handling cases where `size` or `price` might be `None`.
*   The function successfully processed 1000 trade records, generating a structured list of dictionaries with all required fields.
*   The processed trade data was successfully converted into a Pandas DataFrame (`df_trades`) and then saved to a CSV file named `polymarket_trades.csv` without an index.
*   The resulting CSV file includes all extracted and calculated columns, such as `user_id`, `trade_direction`, `shares`, `option_yes_no`, `option_type`, `price`, and `total_trade_amount`, confirming the successful creation and export of structured data.

### Insights or Next Steps

*   The structured trade data, now available in `polymarket_trades.csv`, is ready for in-depth analytical tasks, such as market trend analysis, user behavior segmentation, or performance evaluation of different trade options.
*   Consider automating this data extraction, parsing, and storage process to regularly update the `polymarket_trades.csv` file, providing fresh data for ongoing analysis and reporting.


# Task
Download the `polymarket_trades.csv` file from your Colab environment, and then manually upload it to your GitHub repository.

## Download CSV file from Colab

### Subtask:
Provide code to download the 'polymarket_trades.csv' file directly from the Colab environment to your local machine.


**Reasoning**:
To download the 'polymarket_trades.csv' file from the Colab environment, I will use the `files.download()` function from the `google.colab` module.



In [43]:
from google.colab import files

# Download the CSV file
files.download('polymarket_trades.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Prepare GitHub Repository

### Subtask:
Instruct the user to ensure they have an existing GitHub repository or create a new one where the CSV file will be stored. This step is manual and done outside of the Colab environment.

#### Instructions
1. Go to your GitHub account.
2. Create a new repository if you don't have one already, or choose an existing repository where you want to store the `polymarket_trades.csv` file.
3. Make sure you have appropriate write access to the chosen repository.


## Upload File to GitHub

### Subtask:
Explain the methods for uploading the downloaded 'polymarket_trades.csv' file to the designated GitHub repository, either through the GitHub web interface or using Git commands locally. This step is also manual.


## Summary:

### Data Analysis Key Findings
*   The `polymarket_trades.csv` file was successfully downloaded from the Colab environment to the local machine using the `files.download()` function.
*   Step-by-step instructions were provided for the user to manually prepare a GitHub repository, which included creating a new repository or selecting an existing one, and ensuring appropriate write access.

### Insights or Next Steps
*   The user must manually follow the provided instructions to set up their GitHub repository.
*   The next step in the overall task would involve manually uploading the downloaded `polymarket_trades.csv` file to the prepared GitHub repository.
